In [1]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error,mean_squared_error, r2_score
from xgboost import XGBRegressor


In [2]:
dt = pd.read_csv("Task_Dataset.csv")

In [3]:
label_encoders = {}
categorical_cols = [
    "Priority",
    "Team",
    "Assigned_User",
    "Complexity",

]
for col in categorical_cols:
    le = LabelEncoder()
    dt[col] = le.fit_transform(dt[col])
    label_encoders[col] = le

X = dt[[
    "Priority",
    "Team",
    "Assigned_User",
    "Sprint",
    "Complexity",
    "Experience_Years",
    "Estimated_Hours",
    "Bugs_Reported",
    "Rework_Hours"
]]

y = dt["Actual_Hours"]

x_train, x_test, y_train, y_test = train_test_split(X, y,  test_size= 0.2, random_state =52)

model = XGBRegressor(n_estimators = 200, random_state = 42)
model.fit(x_train,y_train)

predictions = model.predict(x_test)


In [4]:
print(predictions)
print(mean_absolute_error(y_test, predictions))
print(mean_squared_error(y_test,predictions) ** 0.5)
print(r2_score(y_test,predictions))

[30.661291 43.614803 23.387922 25.88304  45.287617 69.95591  87.19004
 72.020584 72.144615 88.4264   23.876484 19.520584 28.481428 54.12927
 24.213428 55.698387 39.509117 15.557333 65.2496   30.883806]
6.262437343597412
8.546007743505196
0.8624098300933838


In [5]:
new_task = pd.DataFrame({

    "Priority":[2],
    "Team":[1],
    "Assigned_User":[3],
    "Sprint":[2],
    "Complexity":[4],
    "Experience_Years":[5],
    "Current_Workload":[8],
    "Estimated_Hours":[40],
    "Bugs_Reported":[2],
    "Rework_Hours":[3]

})

new_task = new_task[X.columns]

predicted_days = model.predict(new_task)

print("Predicted Completion Time:",round(predicted_days[0],2),"days")

Predicted Completion Time: 53.1 days


In [6]:
import pickle
with open("Timeline_Risk_model.pkl", "wb") as file:
    pickle.dump(model,file)

In [7]:
#Overdue Risk Classifier — inputs, algorithm, output


In [ ]:
import pandas as pd

data = {
    "Delay_Days":[0,1,2,4,5,7,8,10],
    "Task_Priority":[1,1,2,2,3,3,3,3],
    "Employee_Workload":[3,4,5,6,8,9,10,12],
    "Escalation_Count":[0,0,1,1,2,2,3,4]
}

df = pd.DataFrame(data)

In [9]:
def risk_label(Delay_Days, Escalation_Count):

    if Delay_Days <= 2 and Escalation_Count == 0:
        return "Low"

    elif Delay_Days <= 5:
        return "Medium"

    else:
        return "High"

In [ ]:
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier

df["Risk"] = df.apply(lambda x: risk_label(x["Delay_Days"], x["Escalation_Count"]),axis=1)

le = LabelEncoder()

df["Risk"] = le.fit_transform(df["Risk"])

X = df[
[
    "Delay_Days",
    "Task_Priority",
    "Employee_Workload",
    "Escalation_Count"
]
]

y = df["Risk"]

model = XGBClassifier()

model.fit(X,y)
print("Model Trained Successfully!")

Model Trained Successfully!


In [ ]:
new_task = pd.DataFrame({

    "Delay_Days":[6],
    "Task_Priority":[3],
    "Employee_Workload":[10],
    "Escalation_Count":[2]

})
pred = model.predict(new_task)

print(le.inverse_transform(pred))

['Medium']


In [12]:
df["Overdue_Risk"] = df.apply(
    lambda row: risk_label(row["Delay_Days"], row["Escalation_Count"]),
    axis=1
)

In [13]:
df["Overdue_Risk"].head()

0       Low
1       Low
2    Medium
3    Medium
4    Medium
Name: Overdue_Risk, dtype: object

In [14]:
le = LabelEncoder()
df["Overdue_Risk"] = le.fit_transform(df["Overdue_Risk"])

In [15]:
df.columns = df.columns.str.strip()
print(df.columns.tolist())

['Delay_Days', 'Task_Priority', 'Assignee_Workload', 'Escalation_Count', 'Risk', 'Overdue_Risk']


In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import mean_absolute_error

target = "Actual_Hours"

X = df[
    [
    "Delay_Days",
    "Task_Priority",
    "Employee_Workload",
    "Escalation_Count"
    ]
]

y = df["Overdue_Risk"]

x_train,x_test,y_train,y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = XGBClassifier(n_estimators=200,random_state=42)

model.fit(x_train,y_train)

pred = model.predict(x_test)



In [17]:
print(pred)
print("mean_absolute_error:",mean_absolute_error(y_test,pred))
print("mean_squared_error:", mean_squared_error(y_test, pred))

[2 0]
mean_absolute_error: 0.5
mean_squared_error: 0.5


In [18]:
#http://localhost:8000/api

In [19]:
tasks = pd.DataFrame({
    "Employee":[
        "Ganesh","Ganesh",
        "Vaibhav","Vaibhav",
        "Mike","Mike"
    ],
    "Completed_On_Time":[
        1,0,
        1,1,
        0,1
    ]
})

sla = tasks.groupby("Employee")["Completed_On_Time"].agg(
    ["sum","count"]
)

sla["SLA_Compliance"] = (
    sla["sum"]/sla["count"]
)*100

print(sla)

          sum  count  SLA_Compliance
Employee                            
Ganesh      1      2            50.0
Mike        1      2            50.0
Vaibhav     2      2           100.0


In [ ]:
sla_payload = {

    "employee":"Ganesh",
    "sla_compliance":100.0,

}

In [21]:
import pickle

with open("prediction_model.pkl", "wb") as file:
    pickle.dump(model,file)

print("Model saved successfully!")

Model saved successfully!


In [22]:
import joblib
from fastapi import FastAPI


joblib.dump(model, "prediction_model.pkl")

app = FastAPI()

allocation_model = joblib.load("prediction_model.pkl")

ModuleNotFoundError: No module named 'fastapi'

In [ ]:
from fastapi import FastAPI
import joblib

app = FastAPI()

model = joblib.load("prediction_model.pkl")

@app.post("/predict-risk")
def predict(data: dict):

    features = [[
        data["days_overdue"],
        data["priority"],
        data["workload"],
        data["escalation_count"]
    ]]

    prediction = model.predict(features)

    return {
        "risk": prediction.tolist()
    }

NameError: name 'Timeline_Risk_model' is not defined